<a href="https://colab.research.google.com/github/eniompw/TinyLM/blob/main/TinyTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets

In [2]:
from datasets import load_dataset
import warnings, itertools
warnings.filterwarnings('ignore')

def load_tinystories(num_stories=200, context_size=4):
    """
    Fetches the TinyStories dataset and prepares it for a character-level language model.
    """
    # Fetch data
    dataset = load_dataset('karpathy/tinystories-gpt4-clean', split='train', streaming=True)
    text = ''.join(s['text'] for s in itertools.islice(dataset, num_stories))

    # Build vocabulary and tokenize
    vocab = sorted(set(text))                                       # ordered list of unique characters
    char_to_id = {c: i for i, c in enumerate(vocab)}                # dictionary mapping char to integer id
    encoded = [char_to_id[c] for c in text]                         # map entire text to integer sequence

    # If context_size=1, skip building windows to save RAM (training loop handles slicing).
    if context_size == 1:
        return [], [], vocab, encoded

    # Create sliding windows for inputs and targets using standard Python lists
    inputs = [encoded[i:i+context_size] for i in range(len(encoded)-context_size)] # sliding windows
    targets = encoded[context_size:]                                               # next char to predict

    return inputs, targets, vocab, encoded

In [3]:
import time, torch
import torch.nn as nn, torch.nn.functional as F
#from tinystories_dataset import load_tinystories

# Automatically create all tensors on GPU if available, removing manual device boilerplate
torch.set_default_device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Hyperparameters ---
context_size = 8                                                                                  # number of previous tokens used to predict next
embed_dim    = 256                                                                                # token/positional embedding dimension (d_model)
n_heads      = 4                                                                                  # number of attention heads in each transformer layer
ffn_dim      = 1024                                                                               # feed-forward network hidden dimension
n_layers     = 2                                                                                  # number of transformer encoder layers
batch_size   = 1024                                                                               # number of samples per training step
lr           = 1e-3                                                                               # initial learning rate
n_steps      = 2001                                                                               # total training steps

# --- Data & Tokenization ---
input_ids, target_ids, idx_to_char, token_ids = load_tinystories(num_stories=1000, context_size=context_size) # previous chars to predict next
input_ids, target_ids = torch.tensor(input_ids), torch.tensor(target_ids)                        # convert to tensors

# --- Model ---
torch.manual_seed(42)                                                                             # seed helper for reproducibility
tok_embed = nn.Embedding(len(idx_to_char), embed_dim)                                            # token embedding lookup layer
pos_embed = nn.Embedding(context_size, embed_dim)                                                # positional embedding for sequence order
transformer = torch.compile(nn.TransformerEncoder(nn.TransformerEncoderLayer(embed_dim, n_heads, ffn_dim, batch_first=True, dropout=0., norm_first=True), n_layers))
linear = nn.Linear(embed_dim, len(idx_to_char))                                                  # maps hidden state to logits (vocab length)

params = list(tok_embed.parameters()) + list(pos_embed.parameters()) + list(transformer.parameters()) + list(linear.parameters())
print(f"params: {sum(p.numel() for p in params):,}")
optimizer = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), fused=True)                     # optimizer replaces manual parameter updates
scheduler, start = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_steps, eta_min=1e-4), time.time() # smoothly decays learning rate

# --- Train ---
for step in range(n_steps):
    batch_idx = torch.randint(0, len(input_ids), (batch_size,))                                  # random batch indices (len(input_ids) is total examples)
    batch_x, batch_y = input_ids[batch_idx], target_ids[batch_idx]                               # fetch random mini-batch (inputs and labels)

    with torch.autocast('cuda', dtype=torch.float16):                                            # float16 mixed precision for speed
        x = tok_embed(batch_x) + pos_embed(torch.arange(context_size))                          # add token and positional embeddings
        loss = F.cross_entropy(linear(transformer(x)[:, -1, :]), batch_y)                        # computes softmax and cross-entropy loss automatically

    optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()                   # zero grads, backprop, AdamW update, and lr decay

    if step % 200 == 0:
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):                       # disable tracking during evaluation
            eval_idx = torch.randint(0, len(input_ids), (4096,))                                 # subset evaluation to prevent GPU OOM
            x_eval = tok_embed(input_ids[eval_idx]) + pos_embed(torch.arange(context_size))     # embed dataset subset
            pred_ids = linear(transformer(x_eval)[:, -1, :]).argmax(1)                          # dataset subset forward & argmax
            print(f"Step {step:4d} | Loss: {loss:.4f} | Acc: {(pred_ids == target_ids[eval_idx]).float().mean():.1%} | {time.time()-start:.1f}s")

print(f"Training time: {time.time() - start:.1f}s")

# --- Generate ---
@torch.no_grad()                                                                                  # disable autograd tracking during inference
def generate(num_chars=200, context_ids=list(token_ids[:context_size])):                         # start with true initial context
    output_chars = [idx_to_char[i] for i in context_ids]                                         # decode initial context to string
    for _ in range(num_chars):
        x = tok_embed(torch.tensor([context_ids])) + pos_embed(torch.arange(context_size))      # embed current window
        next_token_probs = torch.softmax(linear(transformer(x)[:, -1, :]) / 0.7, 1)             # apply temp 0.7 to pick higher-confidence tokens
        next_token = torch.multinomial(next_token_probs, 1).item()                               # randomly sample from predicted distribution
        context_ids, output_chars = context_ids[1:] + [next_token], output_chars + [idx_to_char[next_token]] # slide window forward and append string
    return ''.join(output_chars)

print(generate())

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

params: 1,614,400


W0610 17:21:17.417000 4597 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Step    0 | Loss: 4.7246 | Acc: 19.3% | 26.2s
Step  200 | Loss: 1.4995 | Acc: 53.2% | 28.0s
Step  400 | Loss: 1.4018 | Acc: 59.1% | 29.9s
Step  600 | Loss: 1.3390 | Acc: 61.1% | 31.7s
Step  800 | Loss: 1.1792 | Acc: 63.9% | 33.6s
Step 1000 | Loss: 1.1826 | Acc: 66.1% | 35.4s
Step 1200 | Loss: 1.1419 | Acc: 65.6% | 37.3s
Step 1400 | Loss: 1.1244 | Acc: 66.7% | 39.2s
Step 1600 | Loss: 1.0232 | Acc: 67.4% | 41.0s
Step 1800 | Loss: 1.0561 | Acc: 66.9% | 42.9s
Step 2000 | Loss: 1.0910 | Acc: 67.9% | 44.8s
Training time: 44.8s
Once there.
So, while branch would help his friends to play with his friends and make the fish liked to see the door. They are sun that day laughed they dinner. One day, a little boy named Tim. Tom was very h
